# Airfare Price Prediction EDA

This notebook performs exploratory data analysis on airfare pricing data.

- Data: `data/airfare_sample.csv`
- Report: `reports/eda/eda_report.md`
- Charts directory: `reports/eda`

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path('data/airfare_sample.csv')
OUT_DIR = Path('reports/eda')
OUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET = 'fare_price'

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()

## Data Quality

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing

In [ ]:
plt.figure(figsize=(10,5))
(df.isna().mean() * 100).sort_values(ascending=False).plot(kind='bar', color='#2a9d8f')
plt.title('Missing Value Percentage by Feature')
plt.ylabel('Missing (%)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'missing_values.png', dpi=150)
plt.show()

## Target Distribution

In [ ]:
plt.figure(figsize=(8,5))
df[TARGET].plot(kind='hist', bins=40, color='#457b9d', edgecolor='black')
plt.title('Fare Price Distribution')
plt.xlabel('Fare Price')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fare_distribution.png', dpi=150)
plt.show()

## Correlation Analysis

In [ ]:
corr = df.select_dtypes(include=['number']).corr(numeric_only=True)
plt.figure(figsize=(10,7))
im = plt.imshow(corr, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.index)), corr.index)
plt.title('Correlation Heatmap (Numeric Features)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'correlation_heatmap.png', dpi=150)
plt.show()

corr_target = df.select_dtypes(include=['number']).drop(columns=[TARGET]).corrwith(df[TARGET]).sort_values(key=lambda x: x.abs(), ascending=False)
corr_target

## Feature vs Fare

In [ ]:
features = ['demand_index','seasonality_index','days_to_departure','fuel_cost_index','competitor_price','load_factor']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
for i, col in enumerate(features):
    axes[i].scatter(df[col], df[TARGET], alpha=0.25, s=12)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel(TARGET)
    axes[i].set_title(f'{col} vs {TARGET}')
plt.tight_layout()
plt.savefig(OUT_DIR / 'feature_vs_fare_scatter.png', dpi=150)
plt.show()

## Categorical Impact

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, ['origin', 'destination', 'cabin_class']):
    groups = [g[TARGET].values for _, g in df.groupby(col)]
    labels = [k for k, _ in df.groupby(col)]
    ax.boxplot(groups, tick_labels=labels, showfliers=False)
    ax.set_title(f'{TARGET} by {col}')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(OUT_DIR / 'categorical_boxplots.png', dpi=150)
plt.show()

## Route and Cabin Insights

In [ ]:
cabin_stats = df.groupby('cabin_class')[TARGET].agg(['count','mean','median','min','max']).sort_values('mean', ascending=False)
route_stats = df.groupby(['origin','destination'])[TARGET].agg(['count','mean','median']).sort_values('mean', ascending=False)
print('Fare by cabin class:')
display(cabin_stats.round(2))
print('\nTop expensive routes:')
display(route_stats.head(10).round(2))

## Optional: Load generated markdown report

In [ ]:
report_text = Path('reports/eda/eda_report.md').read_text(encoding='utf-8')
print(report_text[:2000])